# 🔍 Telegram Embeddings Generator
יוצר embeddings ל-500K הודעות לחיפוש סמנטי

**הוראות:**
1. העלה את קובץ `telegram.db` לתיקיית Colab
2. הרץ את כל התאים בסדר
3. הורד את `embeddings.db` בסוף

In [ ]:
# התקנת ספריות
!pip install sentence-transformers tqdm -q
print("✅ ספריות הותקנו")

In [ ]:
# העלאת קובץ DB
from google.colab import files
print("📁 בחר את קובץ telegram.db להעלאה:")
uploaded = files.upload()
print("✅ קובץ הועלה!")

In [ ]:
# טעינת המודל (תומך עברית)
from sentence_transformers import SentenceTransformer
import torch

# בדיקה אם יש GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🖥️ משתמש ב: {device.upper()}")

# מודל רב-לשוני (תומך עברית)
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2', device=device)
print("✅ מודל נטען!")

In [ ]:
# קריאת הודעות מה-DB
import sqlite3

conn = sqlite3.connect('telegram.db')
cursor = conn.execute("""
    SELECT id, from_name, text 
    FROM messages 
    WHERE text IS NOT NULL AND text != '' AND length(text) > 10
""")
messages = cursor.fetchall()
conn.close()

print(f"📊 נמצאו {len(messages):,} הודעות עם טקסט")

In [ ]:
# יצירת Embeddings (החלק שלוקח זמן)
from tqdm import tqdm
import numpy as np

BATCH_SIZE = 256  # גודל batch - מותאם לזיכרון

# הכנת הטקסטים
ids = [m[0] for m in messages]
names = [m[1] or '' for m in messages]
texts = [m[2][:500] for m in messages]  # מקסימום 500 תווים להודעה

# יצירת embeddings ב-batches
all_embeddings = []

print(f"🚀 מייצר embeddings ל-{len(texts):,} הודעות...")
print(f"⏱️ זמן משוער: {len(texts) // 1000} דקות עם GPU, {len(texts) // 200} דקות עם CPU")

for i in tqdm(range(0, len(texts), BATCH_SIZE)):
    batch = texts[i:i+BATCH_SIZE]
    embeddings = model.encode(batch, show_progress_bar=False, convert_to_numpy=True)
    all_embeddings.append(embeddings)

# איחוד כל ה-batches
embeddings_array = np.vstack(all_embeddings)
print(f"✅ נוצרו {embeddings_array.shape[0]:,} embeddings בגודל {embeddings_array.shape[1]}")

In [ ]:
# שמירה לקובץ DB חדש
import struct

# יצירת DB חדש לembeddings
emb_conn = sqlite3.connect('embeddings.db')
emb_conn.execute("""
    CREATE TABLE IF NOT EXISTS embeddings (
        message_id INTEGER PRIMARY KEY,
        from_name TEXT,
        text_preview TEXT,
        embedding BLOB
    )
""")
emb_conn.execute("CREATE INDEX IF NOT EXISTS idx_from_name ON embeddings(from_name)")

# המרת numpy array ל-bytes
def embedding_to_blob(emb):
    return emb.astype(np.float32).tobytes()

# שמירה
print("💾 שומר ל-embeddings.db...")
data = [
    (ids[i], names[i], texts[i][:100], embedding_to_blob(embeddings_array[i]))
    for i in range(len(ids))
]

emb_conn.executemany(
    "INSERT OR REPLACE INTO embeddings VALUES (?, ?, ?, ?)",
    data
)
emb_conn.commit()
emb_conn.close()

print("✅ נשמר בהצלחה!")

In [ ]:
# בדיקת גודל הקובץ
import os
size_mb = os.path.getsize('embeddings.db') / (1024 * 1024)
print(f"📦 גודל הקובץ: {size_mb:.1f} MB")

In [ ]:
# בדיקה מהירה - חיפוש סמנטי
def search_similar(query, top_k=5):
    """חיפוש הודעות דומות לשאילתה"""
    # יצירת embedding לשאילתה
    query_emb = model.encode([query], convert_to_numpy=True)[0]
    
    # טעינת embeddings
    conn = sqlite3.connect('embeddings.db')
    cursor = conn.execute("SELECT message_id, from_name, text_preview, embedding FROM embeddings")
    
    results = []
    for row in cursor:
        msg_id, name, text, emb_blob = row
        emb = np.frombuffer(emb_blob, dtype=np.float32)
        # Cosine similarity
        similarity = np.dot(query_emb, emb) / (np.linalg.norm(query_emb) * np.linalg.norm(emb))
        results.append((similarity, msg_id, name, text))
    
    conn.close()
    
    # מיון לפי דמיון
    results.sort(reverse=True)
    return results[:top_k]

# בדיקה
print("🔍 בדיקת חיפוש סמנטי:")
print("="*50)
query = "איפה אתה עובד?"  # שנה לשאילתה שלך
print(f"שאילתה: {query}\n")

for score, msg_id, name, text in search_similar(query):
    print(f"[{score:.3f}] {name}: {text[:80]}...")
    print()

In [ ]:
# הורדת הקובץ
print("📥 מוריד את embeddings.db...")
files.download('embeddings.db')
print("\n✅ סיום! העבר את הקובץ לתיקיית הפרויקט")